# Comparing Two Percent Accuracy Histograms

In [ ]:
from pointnet_model import pnet
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

#set batch size
BATCH_SIZE = 32

#specify models: 
#M1:
model1_file_stem = 'models/2023-06-14-11:47:00/weights/cp-044.ckpt' 
data1_file_stem = 'voxel_data2/Mg22_size512'

#M2
model2_file_stem = 'models/2023-06-14-11:47:00/weights/cp-038.ckpt' 
data2_file_stem = 'voxel_data2/Mg22_size512'

## Create Percent Accuracy array for Model 1

In [ ]:
# build model
model1 = pnet(sem_seg_flag=True, num_points=512, num_classes=27)
model1.load_weights(model1_file_stem)

 # load test data for model1
test_ds = np.load('{}{}'.format(data1_file_stem, 'test.npy'))
test_features = tf.data.Dataset.from_tensor_slices(test_ds[:, :, :3]).batch(BATCH_SIZE)
test_labels = test_ds[:, :, 3]

# make predictions for model1
predicted_probabilities = model1.predict(test_features)
predictions = np.argmax(predicted_probabilities, axis=2)

# set up name and time for model 1
model1_name = model1_file_stem.split("/")[-1] #modified to save model/ckpt name
model1_time =  model1_file_stem.split("/")[-3] #added to save timestamp
    
#create percent accuracy for m1 
percent_accuracy1 = np.mean(test_labels == predictions, axis=1) #along X axis

## Create Percent Accuracy array for Model 2

In [ ]:
#build model
model2 = pnet(sem_seg_flag=True, num_points=512, num_classes=27)
model2.load_weights(model2_file_stem)

# load test data for model2
test_ds2 = np.load('{}{}'.format(data2_file_stem, 'test.npy'))
test_features2 = tf.data.Dataset.from_tensor_slices(test_ds2[:, :, :3]).batch(BATCH_SIZE)
test_labels2 = test_ds2[:, :, 3]

# make predictions for model2
predicted_probabilities2 = model2.predict(test_features2)
predictions2 = np.argmax(predicted_probabilities2, axis=2)

# evaluate results for model 2
model2_name = model2_file_stem.split("/")[-1] #modified to save model/ckpt name
model2_time =  model2_file_stem.split("/")[-3] #added to save timestamp

# create percent accuracy for model 2 
percent_accuracy2 = np.mean(test_labels2 == predictions2, axis=1) #along X axis

## Plot Overlayed Histograms

In [ ]:
#set title
title = model1_name + model1_time + " vs " + model2_name + model2_time

plt.figure()
plt.hist(percent_accuracy1, bins=100, alpha=0.5, color='purple', label=model1_name)
plt.hist(percent_accuracy2, bins=100, alpha=0.5, color='lime', label=model2_name)
plt.xlabel("Percent accuracy")
plt.ylabel("Frequency")
plt.title(title)
plt.legend(loc = 'upper left')
plt.show()
#plt.savefig("plots/evaluations/comparison_{}.png".format(title)) #to save as png 